# PyTorch, one piece at a time

**ISM 6564 — Week 3 companion · watch-and-type notebook**

The intro notebook (`ism6564-week03-pytorch-intro.ipynb`) built the same MLP
twice — once with sklearn, once with PyTorch — to prove the two are the *same
model*. It moved fast, and several PyTorch ideas went by without being opened
up. This notebook is the slow version: **PyTorch only**, same tickets, same
final model, but every piece is built and tested on its own before it joins
the machine.

The route:

1. **The data, quickly** — the Week 2 prep you already know, compressed
2. **Tensors** — PyTorch's version of numpy arrays
3. **The model, one layer at a time** — `nn.Linear` is a weighted sum you
   have already met, and everything else is stacking
4. **Learning** — the loss, the gradients (autograd), and the optimizer,
   each demonstrated on something tiny before touching the real model
5. **The training loop, in four versions** — five lines first, then
   validation, then mini-batches, then early stopping. By v4 you own,
   line for line, the loop from class
6. **Why you bothered** — four real changes, from a weighted loss to a
   tokenizer you train yourself, each a few lines in *your* loop

You need no PyTorch experience. You do need your sklearn habits — `fit`,
`predict`, train/validation/test — because the whole point is that PyTorch
asks you to build by hand exactly what `MLPClassifier.fit()` was doing in
secret.

---
## Part 0 — Setup

Two libraries do the text preparation (pandas, scikit-learn) and one does
everything after that (torch). We fix every random seed up front so this
notebook prints the same numbers every time it runs — yours included.

In [1]:
import math
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Reproducibility: fix every source of randomness we are about to use.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"pandas  {pd.__version__}")
print(f"torch   {torch.__version__}")

pandas  3.0.2
torch   2.11.0


---
## Part 1 — The data, quickly *(this part is Week 2 material, not the lesson)*

Same corpus as your assignment: Northwind support tickets, each labelled with
the queue a human dispatcher routed it to. Our model's job is to predict that
queue from the ticket's text.

The preparation below is everything Weeks 2–3 already taught, run at speed:
look at the data, drop duplicate resubmissions, make **three splits**, and
turn text into TF-IDF numbers **fitted on the training split only**. The
three splits each have one job:

- **train** — the model learns from these tickets;
- **validation** — checked during training, to make decisions (like when to
  stop);
- **test** — touched exactly once, at the very end, for the honest number.

If any line in this part is unfamiliar, the lecture notebook covers it in
full. The lesson of *this* notebook starts at Part 2.

In [2]:
DATA_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-03-text-classification/data/tickets.csv")

tickets = pd.read_csv(DATA_URL).fillna("")

print(f"{len(tickets)} tickets\n")
print(tickets["queue"].value_counts())

3000 tickets

queue
shipping            890
returns-warranty    637
billing             589
product-question    531
account-access      353
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

# The text a model sees: subject and body glued together (Week 2's choice).
tickets["text"] = (tickets["subject"] + " " + tickets["body"]).str.strip()

# Dispatchers under load resubmit tickets. Drop exact duplicates BEFORE
# splitting, so no ticket can sit in train and test at the same time.
tickets = tickets.drop_duplicates("body").reset_index(drop=True)

# Three stratified splits: 60% train, 20% validation, 20% test.
train_df, temp_df = train_test_split(
    tickets, test_size=0.40, stratify=tickets["queue"], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["queue"], random_state=SEED)

print(f"train {len(train_df)}   val {len(val_df)}   test {len(test_df)}")

train 1758   val 586   test 586


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Which tokenizer? We pass none, so sklearn uses its DEFAULT: lowercase the
# text, then pull out tokens with the regex r"(?u)\\b\\w\\w+\\b" -- runs of 2+
# letters/digits. Punctuation is discarded, 1-character tokens are dropped,
# and nothing else happens (no stopword removal, no stemming, no subwords):
#   "Order #8841 arrived DAMAGED, I want a refund!"
#     -> ['order', '8841', 'arrived', 'damaged', 'want', 'refund']
vec = TfidfVectorizer(min_df=3)          # keep words appearing in >= 3 tickets
X_train = vec.fit_transform(train_df["text"])   # fit + transform: TRAIN ONLY
X_val   = vec.transform(val_df["text"])         # transform: same columns
X_test  = vec.transform(test_df["text"])

le = LabelEncoder().fit(train_df["queue"])      # queue names -> integers 0..4
y_train = le.transform(train_df["queue"])
y_val   = le.transform(val_df["queue"])
y_test  = le.transform(test_df["queue"])

N_FEATURES = X_train.shape[1]
N_CLASSES  = len(le.classes_)

# A surprise worth meeting on purpose: X_train is NOT a numpy array. TF-IDF
# output is almost all zeros, so sklearn returns a scipy SPARSE matrix -- it
# stores only the non-zero values plus their (row, column) positions. Same
# numbers, a far thriftier container: that is what makes wide text matrices
# affordable at all. (Part 2 converts it for PyTorch.)
print(type(X_train), "\n")

print(f"X_train {X_train.shape}   X_val {X_val.shape}   X_test {X_test.shape}")
print(f"\n{N_FEATURES} features  ->  {N_CLASSES} classes: {list(le.classes_)}")

<class 'scipy.sparse._csr.csr_matrix'> 

X_train (1758, 636)   X_val (586, 636)   X_test (586, 636)

636 features  ->  5 classes: ['account-access', 'billing', 'product-question', 'returns-warranty', 'shipping']


**Checkpoint.** Every ticket is now a row of 636 numbers, and every label is
an integer 0–4. That is all a model ever sees. The sklearn part of the day is
over — from here on, everything is PyTorch.

---
## Part 2 — Tensors: PyTorch's numpy

A **tensor** is PyTorch's array. If you can use a numpy array, you can use a
tensor — same shapes, same indexing, same elementwise arithmetic. PyTorch has
its own array type for two reasons:

1. tensors can live on a GPU (matters in later weeks, not today), and
2. tensors can **remember the arithmetic done to them**, which is what makes
   automatic gradients possible in Part 4 — the one genuinely new idea in
   this whole notebook.

First, prove to yourself it behaves like numpy.

In [5]:
t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print("shape :", t.shape)          # like arr.shape
print("dtype :", t.dtype)          # like arr.dtype
print("t * 10:") ; print(t * 10)   # elementwise, like numpy
print("column sums:", t.sum(dim=0))  # numpy's axis= is torch's dim=

shape : torch.Size([2, 3])
dtype : torch.float32
t * 10:
tensor([[10., 20., 30.],
        [40., 50., 60.]])
column sums: tensor([5., 7., 9.])


In [6]:
# numpy <-> torch is a cheap, common round trip:
arr  = np.array([1.0, 2.0, 3.0])
tens = torch.from_numpy(arr)        # numpy -> tensor
back = tens.numpy()                 # tensor -> numpy

print(type(arr).__name__, "->", type(tens).__name__, "->", type(back).__name__)

# One habit to build now: neural networks compute in float32 ("single
# precision"), not numpy's float64 default. Half the memory, plenty of
# precision for gradient-based learning, and it is torch's default:
print("torch default:", torch.tensor([1.0, 2.0]).dtype)
print("numpy default:", np.array([1.0, 2.0]).dtype)

ndarray -> Tensor -> ndarray
torch default: torch.float32
numpy default: float64


Now convert our real data. Two dtype rules to memorize, because the error
messages when you break them are unfriendly:

- **features** must be `float32` — the network's arithmetic runs in single
  precision;
- **class labels** must be `int64` (torch calls it `long`) — the loss
  function in Part 4 insists on it.

One honest caveat: `.toarray()` **densifies** the sparse matrix. Recall what
"sparse" buys (Part 1 printed the type): a `csr_matrix` stores only the
non-zero values and their positions, so a mostly-zero matrix costs almost
nothing. `.toarray()` undoes that — every zero becomes a real, stored number.
At 636 columns that is small and safe. At Week 2's 50,000-column matrices it
was a `MemoryError` — which is exactly why the lecture compressed with SVD
before going dense. Here `min_df=3` already did the shrinking for us.

In [7]:
def to_tensor(X):
    """scipy sparse matrix -> dense float32 tensor."""
    return torch.from_numpy(X.toarray().astype(np.float32))


X_train_t = to_tensor(X_train)
X_val_t   = to_tensor(X_val)
X_test_t  = to_tensor(X_test)

y_train_t = torch.from_numpy(y_train).long()   # int64 class indices
y_val_t   = torch.from_numpy(y_val).long()
y_test_t  = torch.from_numpy(y_test).long()

print(f"X_train_t {tuple(X_train_t.shape)}   dtype {X_train_t.dtype}")
print(f"y_train_t {tuple(y_train_t.shape)}   dtype {y_train_t.dtype}")

X_train_t (1758, 636)   dtype torch.float32
y_train_t (1758,)   dtype torch.int64


---
## Part 3 — The model, one layer at a time

In sklearn the whole model was one line:

```python
MLPClassifier(hidden_layer_sizes=(128,))
```

That line is a closed box. We are now going to build the box's contents by
hand, starting from its only real component: the **`nn.Linear`** layer.

Start tiny — 4 inputs, 3 outputs — so we can see every number.

In [8]:
torch.manual_seed(SEED)            # so your random layer matches ours

layer = nn.Linear(4, 3)            # 4 numbers in -> 3 numbers out

# A Linear layer owns exactly two things, and both start as small random
# numbers that training will later adjust:
print("weight:", tuple(layer.weight.shape))    # one row of 4 weights per output
print(layer.weight.data)
print("\nbias  :", tuple(layer.bias.shape))    # one bias per output
print(layer.bias.data)

# Push two fake "tickets" (4 features each) through it:
x = torch.tensor([[1.0, 0.0, 2.0, 0.0],
                  [0.0, 3.0, 0.0, 1.0]])
print("\nlayer(x):") ; print(layer(x))          # 2 rows in -> 2 rows of 3 out

weight: (3, 4)
tensor([[ 0.3823,  0.4150, -0.1171,  0.4593],
        [-0.1096,  0.1009, -0.2434,  0.2936],
        [ 0.4408, -0.3668,  0.4346,  0.0936]])

bias  : (3,)
tensor([0.3694, 0.0677, 0.2411])

layer(x):
tensor([[ 0.5174, -0.5287,  1.5511],
        [ 2.0737,  0.6640, -0.7658]], grad_fn=<AddmmBackward0>)


**What did it just compute? Nothing you have not already met.** For each
output, the layer multiplies every input by a weight and adds them up, plus a
bias — a **weighted sum**. That is the same arithmetic as the logistic
regression whose coefficients we read in the lecture (all 51,840 of them).
Prove it by doing the layer's job manually:

In [9]:
manual = x @ layer.weight.T + layer.bias     # weighted sums, written by hand

print(manual)
print("\nsame as layer(x)?", torch.allclose(manual, layer(x)))

tensor([[ 0.5174, -0.5287,  1.5511],
        [ 2.0737,  0.6640, -0.7658]], grad_fn=<AddBackward0>)

same as layer(x)? True


So one `nn.Linear` is (multi-output) logistic regression's engine. Then why
stack two? Because **two weighted sums in a row collapse into one** — sums of
sums are still sums, so `Linear -> Linear` has exactly the power of a single
`Linear`, just with extra steps.

The fix is one non-linear function between them: **ReLU**, which replaces
every negative number with zero and leaves positives alone. It is the
cheapest possible kink, and a kink is all it takes to stop the collapse —
with it, the first layer can build intermediate features and the second can
combine them.

In [10]:
before = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
after  = torch.relu(before)

print("before relu:", before)
print("after  relu:", after)       # negatives clipped to zero, that is all

before relu: tensor([-2.0000, -0.5000,  0.0000,  0.5000,  2.0000])
after  relu: tensor([0.0000, 0.0000, 0.0000, 0.5000, 2.0000])


Now assemble the real thing. **`nn.Sequential` means "run these in order"** —
it is the model-building equivalent of a recipe. Our sizes: 636 TF-IDF
features in, a hidden layer of 128 (matching the intro notebook and sklearn's
`hidden_layer_sizes=(128,)`), 5 queue scores out.

In [11]:
torch.manual_seed(SEED)            # random initial weights, reproducibly

model = nn.Sequential(
    nn.Linear(N_FEATURES, 128),    # 636 -> 128   learn intermediate features
    nn.ReLU(),                     #              the kink (owns no numbers)
    nn.Linear(128, N_CLASSES),     # 128 -> 5     combine them into 5 scores
)

print(model)

Sequential(
  (0): Linear(in_features=636, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=5, bias=True)
)


In [12]:
# Audit every number this model owns. "Parameter" = a number training may
# change. Each Linear(in, out) owns an out-x-in weight matrix + out biases.
total = 0
for name, p in model.named_parameters():
    print(f"{name:10s} {str(tuple(p.shape)):12s} {p.numel():7,d} numbers")
    total += p.numel()

print(f"\ntotal: {total:,} parameters")
print(f"check: 636*128 + 128 + 128*5 + 5 = {636*128 + 128 + 128*5 + 5:,}")

0.weight   (128, 636)    81,408 numbers
0.bias     (128,)           128 numbers
2.weight   (5, 128)         640 numbers
2.bias     (5,)               5 numbers

total: 82,181 parameters
check: 636*128 + 128 + 128*5 + 5 = 82,181


82,181 numbers, all currently random — which means the model already *runs*,
it just runs badly. Push three real tickets through it and look at what comes
out. The five raw output scores are called **logits**: bigger means "more
likely this queue", but they are not probabilities — they can be negative,
and they do not sum to one.

In [13]:
logits = model(X_train_t[:3])      # 3 tickets in -> 3 rows of 5 scores out

print("queues:", list(le.classes_), "\n")
print(logits)

queues: ['account-access', 'billing', 'product-question', 'returns-warranty', 'shipping'] 

tensor([[ 0.0874, -0.0020,  0.0110,  0.0433,  0.0622],
        [ 0.0969,  0.0018,  0.0113,  0.0485,  0.0663],
        [ 0.0936, -0.0004,  0.0109,  0.0428,  0.0819]],
       grad_fn=<AddmmBackward0>)


In [14]:
# softmax turns each row of logits into probabilities (positive, sums to 1).
probs = torch.softmax(logits, dim=1)

print(probs, "\n")
print("each row sums to:", probs.sum(dim=1))
print("\npredicted:", [le.classes_[i] for i in probs.argmax(dim=1)])
print("actual   :", list(train_df["queue"].iloc[:3]))

tensor([[0.2095, 0.1916, 0.1941, 0.2005, 0.2043],
        [0.2105, 0.1914, 0.1933, 0.2006, 0.2042],
        [0.2096, 0.1908, 0.1930, 0.1993, 0.2072]], grad_fn=<SoftmaxBackward0>) 

each row sums to: tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)

predicted: ['account-access', 'account-access', 'account-access']
actual   : ['product-question', 'returns-warranty', 'account-access']


Roughly a fifth of the probability on each queue — an untrained model is a
5-sided die. Everything from here to the end of Part 5 is about turning those
82,181 random numbers into informed ones.

---
## Part 4 — Learning: the loss, the gradients, the optimizer

Training answers three questions, over and over:

1. **How wrong are we?** — the *loss function* boils all predictions down to
   one number.
2. **Which direction should each weight move to be less wrong?** — the
   *gradients*, computed for us by autograd.
3. **Who actually moves them?** — the *optimizer*.

One piece at a time.

In [15]:
# CrossEntropyLoss is THE loss for classification. Two things it insists on,
# both common sources of beginner errors:
#   1. give it raw LOGITS  -- it applies softmax itself; do not softmax first
#   2. give it integer class labels (our int64 tensor) -- not one-hot columns
loss_fn = nn.CrossEntropyLoss()

loss = loss_fn(model(X_train_t), y_train_t)   # untrained model, whole train set

print(f"untrained loss: {loss.item():.4f}")
print(f"know-nothing benchmark ln(5) = {math.log(5):.4f}")

untrained loss: 1.6101
know-nothing benchmark ln(5) = 1.6094


The loss of a model that spreads its probability evenly over 5 classes is
exactly ln(5) ≈ 1.609 — and our untrained model sits right at it, as it
should. Perfect predictions would score 0. So training means: **push this
one number down.**

### The one piece of real magic: autograd

To push the loss down, we need to know — for *each* of the 82,181 weights —
whether nudging it up or down lowers the loss, and by how much. That is a
derivative, and PyTorch computes all of them with one method call, because
tensors remember the arithmetic that produced them.

Watch it on a single number first, where we can check by hand. For
`y = x²`, calculus says the slope is `2x` — at `x = 3`, the slope is 6.

In [16]:
x = torch.tensor(3.0, requires_grad=True)   # "track what happens to x"

y = x ** 2                # y remembers it was built by squaring x
y.backward()              # "compute the slope of y with respect to x"

print("slope stored in x.grad:", x.grad)    # 2x = 6, no calculus typed by us

slope stored in x.grad: tensor(6.)


`x.grad = 6` reads: *near x = 3, increasing x makes y grow 6 times as fast* —
so to make y smaller, move x **against** the gradient. That is the entire
idea of training, and it scales without modification: call `.backward()` on
the loss, and every parameter in the model gets its own `.grad`.

In [17]:
loss = loss_fn(model(X_train_t), y_train_t)   # measure again (fresh graph)
loss.backward()                               # slopes for ALL 82,181 numbers

first_layer = model[0]                        # Sequential is indexable
print("weight shape:", tuple(first_layer.weight.shape))
print("grad   shape:", tuple(first_layer.weight.grad.shape))   # one slope per weight

print("\na few of the slopes:")
print(first_layer.weight.grad[0, :5])

weight shape: (128, 636)
grad   shape: (128, 636)

a few of the slopes:
tensor([-3.9886e-05, -3.3317e-05,  0.0000e+00, -4.3522e-07,  0.0000e+00])


One trap before we automate this: **gradients accumulate.** A second
`.backward()` does not replace `.grad` — it *adds to it*. Back to the tiny
example to see it happen:

In [18]:
x = torch.tensor(3.0, requires_grad=True)

(x ** 2).backward()
print("after one backward :", x.grad)     # 6  -- correct

(x ** 2).backward()
print("after two backwards:", x.grad)     # 12 -- 6 + 6, NOT the true slope

x.grad = None                             # what zeroing fixes
(x ** 2).backward()
print("zero, then backward:", x.grad)     # 6  -- correct again

after one backward : tensor(6.)
after two backwards: tensor(12.)
zero, then backward: tensor(6.)


Accumulation exists for advanced uses; for us it is a bug waiting to happen.
Rule: **zero the gradients before every new measurement.** The optimizer has
a method for exactly that, which brings us to question 3.

### The optimizer

The optimizer's whole job: given the gradients, nudge every weight a small
step against its slope. We hand it the model's parameters — "these are the
numbers you are allowed to change" — and accept **every default** of the
standard choice, Adam. (Its most important default: learning rate `lr=0.001`,
the step size. Part 6 says more.)

Here is **one complete heartbeat of training**, the five steps you are about
to see thousands of times:

In [19]:
optimizer = torch.optim.Adam(model.parameters())   # all defaults

before = loss_fn(model(X_train_t), y_train_t).item()

optimizer.zero_grad()                        # 1. clear old gradients
predictions = model(X_train_t)               # 2. forward pass
loss = loss_fn(predictions, y_train_t)       # 3. measure the error
loss.backward()                              # 4. compute all 82,181 slopes
optimizer.step()                             # 5. nudge every weight

after = loss_fn(model(X_train_t), y_train_t).item()

print(f"loss before the step: {before:.4f}")
print(f"loss after  the step: {after:.4f}   (lower -- it learned, a little)")

loss before the step: 1.6101
loss after  the step: 1.6049   (lower -- it learned, a little)


That is it. There is no sixth step. All that remains is doing this many
times, and being scientific about when to stop — which is Part 5.

---
## Part 5 — The training loop, in four versions

sklearn's `.fit()` is a loop like the one you just met, run until a stopping
rule fires, with the bookkeeping hidden. We will build it in four versions,
each fixing the previous one's biggest problem. **The five numbered lines
never change** — versions only add measurement and judgement around them.

### v1 — the minimal loop

Repeat the heartbeat for a fixed number of epochs. An **epoch** = one full
pass over the training data. We rebuild the model from the same seed so every
version starts from identical random weights — fair comparisons.

In [20]:
torch.manual_seed(SEED)                          # identical starting weights
model_v1 = nn.Sequential(
    nn.Linear(N_FEATURES, 128), nn.ReLU(), nn.Linear(128, N_CLASSES))
optimizer = torch.optim.Adam(model_v1.parameters())

for epoch in range(1, 31):
    optimizer.zero_grad()                        # 1. clear old gradients
    predictions = model_v1(X_train_t)            # 2. forward pass
    loss = loss_fn(predictions, y_train_t)       # 3. measure the error
    loss.backward()                              # 4. compute all gradients
    optimizer.step()                             # 5. update the weights

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:3d}   loss {loss.item():.4f}")

epoch   1   loss 1.6101
epoch   5   loss 1.5892
epoch  10   loss 1.5561
epoch  15   loss 1.5091
epoch  20   loss 1.4487
epoch  25   loss 1.3770
epoch  30   loss 1.2952


The loss falls — the model is learning. But v1 is flying blind in one
important way: the loss is measured on the **training** tickets, the ones the
model is memorizing. It cannot tell us the only thing we care about — does
this model work on tickets it has *not* seen?

For that we need an accuracy measure, run on the validation split. Two new
PyTorch idioms appear here, both about *measuring without learning*:

- **`model.eval()` / `model.train()`** — switches layers that behave
  differently during evaluation (none in this model yet; Part 6's dropout is
  the classic example). Free to call now, a bug-preventer later.
- **`torch.no_grad()`** — "do not record arithmetic for gradients here."
  Measurement needs no slopes, so this saves time and memory.

In [21]:
def accuracy(model, X, y):
    """Fraction of correct predictions -- measuring only, no learning."""
    model.eval()                             # evaluation mode
    with torch.no_grad():                    # no gradient bookkeeping
        predicted = model(X).argmax(dim=1)   # biggest logit = predicted class
    model.train()                            # back to training mode
    return (predicted == y).float().mean().item()


print(f"v1  train accuracy: {accuracy(model_v1, X_train_t, y_train_t):.4f}")
print(f"v1  val   accuracy: {accuracy(model_v1, X_val_t,   y_val_t):.4f}")

v1  train accuracy: 0.8396
v1  val   accuracy: 0.8157


Training accuracy is higher than validation accuracy — the model is always
better on tickets it studied. The *validation* number is the one that
predicts real performance, so:

### v2 — measure validation every epoch

In [22]:
torch.manual_seed(SEED)                          # same starting weights again
model_v2 = nn.Sequential(
    nn.Linear(N_FEATURES, 128), nn.ReLU(), nn.Linear(128, N_CLASSES))
optimizer = torch.optim.Adam(model_v2.parameters())

for epoch in range(1, 151):
    optimizer.zero_grad()                        # the same five lines...
    predictions = model_v2(X_train_t)
    loss = loss_fn(predictions, y_train_t)
    loss.backward()
    optimizer.step()

    val_acc = accuracy(model_v2, X_val_t, y_val_t)   # ...plus one measurement

    if epoch == 1 or epoch % 15 == 0:
        print(f"epoch {epoch:3d}   loss {loss.item():.4f}   val accuracy {val_acc:.4f}")

epoch   1   loss 1.6101   val accuracy 0.2167
epoch  15   loss 1.5091   val accuracy 0.5563
epoch  30   loss 1.2952   val accuracy 0.8157
epoch  45   loss 1.0052   val accuracy 0.8652
epoch  60   loss 0.7069   val accuracy 0.9061
epoch  75   loss 0.4788   val accuracy 0.9096
epoch  90   loss 0.3413   val accuracy 0.9096
epoch 105   loss 0.2640   val accuracy 0.9130
epoch 120   loss 0.2171   val accuracy 0.9147
epoch 135   loss 0.1851   val accuracy 0.9130
epoch 150   loss 0.1613   val accuracy 0.9147


Now we can *see* what the loss alone hid. From epoch 75 to 150 the training
loss falls from 0.48 to 0.16 — a huge apparent improvement — while validation
accuracy just sits parked around 0.91. Past that point the model is polishing
its memory of the training set, not getting better at the job. Two problems remain, and each gets a version:

- v2 takes **one weight update per epoch** — the whole training set is
  averaged into a single step. Real datasets do not fit in memory at once,
  and many small steps learn more per pass than one big one. → **v3:
  mini-batches.**
- v2 trains for a **fixed number of epochs** picked by us, and keeps whatever
  weights it happens to end on. → **v4: stop when validation stops
  improving, and keep the best weights seen.**

### v3 — mini-batches

`DataLoader` is PyTorch's batching machine: give it the (X, y) pairs and a
batch size, and it deals shuffled hands of 200 tickets each epoch.

In [23]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_t, y_train_t)   # zips X and y together
train_dl = DataLoader(train_ds, batch_size=200, shuffle=True)

print(f"{len(train_ds)} training tickets -> {len(train_dl)} batches of up to 200")

xb, yb = next(iter(train_dl))                    # peek at one batch
print(f"one batch: X {tuple(xb.shape)}, y {tuple(yb.shape)}")

1758 training tickets -> 9 batches of up to 200
one batch: X (200, 636), y (200,)


In [24]:
torch.manual_seed(SEED)                          # same starting weights again
model_v3 = nn.Sequential(
    nn.Linear(N_FEATURES, 128), nn.ReLU(), nn.Linear(128, N_CLASSES))
optimizer = torch.optim.Adam(model_v3.parameters())

for epoch in range(1, 31):
    for xb, yb in train_dl:                      # NEW: one hand at a time
        optimizer.zero_grad()                    # the same five lines,
        predictions = model_v3(xb)               # now on 200-ticket batches
        loss = loss_fn(predictions, yb)
        loss.backward()
        optimizer.step()

    val_acc = accuracy(model_v3, X_val_t, y_val_t)

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:3d}   loss {loss.item():.4f}   val accuracy {val_acc:.4f}")

epoch   1   loss 1.5783   val accuracy 0.3123


epoch   5   loss 1.1402   val accuracy 0.8225


epoch  10   loss 0.4713   val accuracy 0.9096
epoch  15   loss 0.2644   val accuracy 0.9130
epoch  20   loss 0.2034   val accuracy 0.9164
epoch  25   loss 0.1177   val accuracy 0.9147
epoch  30   loss 0.1581   val accuracy 0.9198


Nine updates per epoch instead of one, and the model reaches its plateau in
a fraction of the epochs: v3 is at 0.9096 by epoch **10**, a level v2 needed
about **75** epochs to reach. (The printed loss is now just the *last
batch's* loss — a noisier number than v2's, which is fine; validation
accuracy is the reading that matters.)

### v4 — early stopping: the loop from class

Last problems: *when to stop*, and *which weights to keep*. The standard
answer to both at once:

- after each epoch, if validation improved, **remember a copy of the current
  weights** (`state_dict()` is the model's numbers as a dictionary);
- if it has not improved for `PATIENCE` epochs in a row, stop, and **roll
  back** to the best copy.

Patience is a judgement call: too small and you quit on a temporary dip in a
bumpy curve; too large and you train pointlessly. We use 10, matching the
lecture's walkthrough.

This is `MLPClassifier(early_stopping=True)`, built from parts you now
recognize individually.

In [25]:
torch.manual_seed(SEED)                          # same starting weights again
model_v4 = nn.Sequential(
    nn.Linear(N_FEATURES, 128), nn.ReLU(), nn.Linear(128, N_CLASSES))
optimizer = torch.optim.Adam(model_v4.parameters())

MAX_EPOCHS, PATIENCE = 200, 10
best_val, best_weights, epochs_without_improvement = 0.0, None, 0

for epoch in range(1, MAX_EPOCHS + 1):
    for xb, yb in train_dl:
        optimizer.zero_grad()                    # 1. clear old gradients
        predictions = model_v4(xb)               # 2. forward pass
        loss = loss_fn(predictions, yb)          # 3. measure the error
        loss.backward()                          # 4. compute all gradients
        optimizer.step()                         # 5. update the weights

    val_acc = accuracy(model_v4, X_val_t, y_val_t)

    if val_acc > best_val:                       # improved: remember weights
        best_val = val_acc
        best_weights = {k: v.clone() for k, v in model_v4.state_dict().items()}
        epochs_without_improvement = 0
    else:                                        # no improvement: done yet?
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print(f"early stop at epoch {epoch}")
            break

    if epoch == 1 or epoch % 5 == 0:
        print(f"epoch {epoch:3d}   loss {loss.item():.4f}   val accuracy {val_acc:.4f}")

model_v4.load_state_dict(best_weights)           # roll back to the best epoch
print(f"\nbest validation accuracy: {best_val:.4f}")

epoch   1   loss 1.5783   val accuracy 0.3123
epoch   5   loss 1.1402   val accuracy 0.8225


epoch  10   loss 0.4713   val accuracy 0.9096


epoch  15   loss 0.2644   val accuracy 0.9130


epoch  20   loss 0.2034   val accuracy 0.9164
epoch  25   loss 0.1177   val accuracy 0.9147
epoch  30   loss 0.1581   val accuracy 0.9198
epoch  35   loss 0.1389   val accuracy 0.9181


early stop at epoch 39

best validation accuracy: 0.9198


**You now own every line of this loop.** It is, character for character, the
loop from the lecture and the intro notebook — the one that looked like a
wall of code an hour ago.

One thing left. The validation set chose when to stop and which weights to
keep, so its score is slightly flattered — it was consulted dozens of times.
The **test set** has been touched by nothing. Spend it, once:

In [26]:
from sklearn.metrics import f1_score

model_v4.eval()
with torch.no_grad():
    test_pred = model_v4(X_test_t).argmax(dim=1).numpy()

print(f"TEST accuracy: {(test_pred == y_test).mean():.4f}")
print(f"TEST macro F1: {f1_score(y_test, test_pred, average='macro'):.4f}")

TEST accuracy: 0.8959
TEST macro F1: 0.8879


---
## Part 6 — Why you bothered: the loop is now yours to change

Everything above, sklearn does in one line — so why build it by hand? Because
sklearn's box is closed, and yours is now open. Each change below is a few
lines *in code you understand*, and the first and last are things
`MLPClassifier` **cannot do at all**.

First, stop retyping: a loop you understand may be wrapped in a function.
Nothing inside `fit()` is new — it is v4, verbatim, with the model and loss
passed in.

In [27]:
def make_model(hidden=128, dropout=0.0, n_features=None):
    """Fresh model from the same seed -- every variant starts identically."""
    torch.manual_seed(SEED)
    n_in = n_features if n_features is not None else N_FEATURES
    layers = [nn.Linear(n_in, hidden), nn.ReLU()]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))     # explained below
    layers.append(nn.Linear(hidden, N_CLASSES))
    return nn.Sequential(*layers)


def fit(model, loss_fn=nn.CrossEntropyLoss(), max_epochs=200, patience=10,
        data=None, X_val=None, y_val=None):
    """The v4 loop, verbatim, as a reusable tool. The data is a parameter
    too, defaulting to our tensors -- the loop does not care where the
    numbers came from. Change 4 uses exactly that."""
    data  = data  if data  is not None else train_dl
    X_val = X_val if X_val is not None else X_val_t
    y_val = y_val if y_val is not None else y_val_t
    optimizer = torch.optim.Adam(model.parameters())
    best_val, best_weights, without_improvement = 0.0, None, 0
    for epoch in range(1, max_epochs + 1):
        for xb, yb in data:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        val_acc = accuracy(model, X_val, y_val)
        if val_acc > best_val:
            best_val = val_acc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            without_improvement = 0
        else:
            without_improvement += 1
            if without_improvement >= patience:
                break
    model.load_state_dict(best_weights)
    return best_val, epoch


def val_macro_f1(model, X=None, labels=None):
    X = X if X is not None else X_val_t
    labels = labels if labels is not None else y_val
    model.eval()
    with torch.no_grad():
        pred = model(X).argmax(dim=1).numpy()
    return f1_score(labels, pred, average="macro")


# Sanity check: same seed, same loop -- fit() must reproduce v4 exactly.
baseline = make_model()
base_val, base_epochs = fit(baseline)
print(f"baseline: val accuracy {base_val:.4f} in {base_epochs} epochs")
print(f"v4 found: val accuracy {best_val:.4f} -- the function IS the loop")

results = [{"variant": "baseline (v4)", "val_accuracy": base_val,
            "val_macro_F1": val_macro_f1(baseline)}]

baseline: val accuracy 0.9198 in 39 epochs
v4 found: val accuracy 0.9198 -- the function IS the loop


### Change 1 — class weights: the one sklearn's MLP cannot make

Our queues are mildly imbalanced — shipping has about 2.5× the tickets of
account-access — and Week 3's evaluation block showed what models do to small
classes when imbalance gets severe: quietly sacrifice them, while accuracy
looks fine.
The standard treatment is to make mistakes on rare classes **cost more** in
the loss. In your loop that is one argument — `CrossEntropyLoss(weight=...)`
— using the same "balanced" formula as sklearn's `class_weight="balanced"`.
`MLPClassifier` simply has no such knob; this is the first concrete thing
your open box can do that the closed one cannot.

In [28]:
counts = np.bincount(y_train)                       # tickets per queue (train)
w = len(y_train) / (N_CLASSES * counts)             # rare class -> big weight
class_weights = torch.tensor(w, dtype=torch.float32)

for q, n, wt in zip(le.classes_, counts, w):
    print(f"{q:18s} {n:4d} tickets   weight {wt:.2f}")

weighted = make_model()          # identical starting weights, same batches
wt_val, _ = fit(weighted, loss_fn=nn.CrossEntropyLoss(weight=class_weights))

results.append({"variant": "class-weighted loss", "val_accuracy": wt_val,
                "val_macro_F1": val_macro_f1(weighted)})
print(f"\nclass-weighted: val accuracy {wt_val:.4f}   "
      f"val macro F1 {val_macro_f1(weighted):.4f}")

account-access      207 tickets   weight 1.70
billing             349 tickets   weight 1.01
product-question    310 tickets   weight 1.13
returns-warranty    372 tickets   weight 0.95
shipping            520 tickets   weight 0.68



class-weighted: val accuracy 0.9164   val macro F1 0.9082


### Change 2 — dropout: a layer that only exists during training

`nn.Dropout(0.3)` randomly silences 30% of the hidden units on every training
batch, forcing the survivors to carry the signal — the classic guard against
over-relying on any one feature. During evaluation it switches itself off and
lets everything through.

This is why `model.train()` / `model.eval()` were worth wiring in early: our
`accuracy()` helper already flips the switch correctly, so dropout drops into
the pipeline with **zero changes to the loop**.

In [29]:
dropped = make_model(dropout=0.3)
print(dropped, "\n")

dr_val, _ = fit(dropped)

results.append({"variant": "dropout 0.3", "val_accuracy": dr_val,
                "val_macro_F1": val_macro_f1(dropped)})
print(f"dropout: val accuracy {dr_val:.4f}   val macro F1 {val_macro_f1(dropped):.4f}")

Sequential(
  (0): Linear(in_features=636, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=128, out_features=5, bias=True)
) 

dropout: val accuracy 0.9181   val macro F1 0.9111


### Change 3 — capacity is a number you type

Want a wider hidden layer? Change `128`. A deeper model? Add a
`Linear -> ReLU` pair to the list. In sklearn that was
`hidden_layer_sizes=`; here it is the same idea, except you can now *mix* it
with everything else (dropout between specific layers, different activations,
a frozen layer — Week 4 will add new layer *types* to this same recipe).

In [30]:
wide = make_model(hidden=512)                       # 4x the hidden units
n_params = sum(p.numel() for p in wide.parameters())
print(f"hidden 512 -> {n_params:,} parameters (baseline was 82,181)\n")

wide_val, _ = fit(wide)

results.append({"variant": "hidden = 512", "val_accuracy": wide_val,
                "val_macro_F1": val_macro_f1(wide)})
print(f"hidden 512: val accuracy {wide_val:.4f}   val macro F1 {val_macro_f1(wide):.4f}")

hidden 512 -> 328,709 parameters (baseline was 82,181)



hidden 512: val accuracy 0.9198   val macro F1 0.9128


### Change 4 — swap the tokenizer: the prep is yours to change too

Changes 1–3 turned model-side knobs. The prep has knobs too — and the biggest
one is hiding in Part 1's note: our `TfidfVectorizer` splits text with its
default word regex. That has a real cost on this corpus, where **a quarter of
the tickets contain typos**: a one-off misspelling like `trackign` is a
token the regex has seen once, `min_df=3` throws it away, and the word
contributes *nothing* to the ticket's row.

Week 1's answer was **subword tokenization**. Train a byte-level BPE — the
same recipe as Week 1, and on the **training tickets only**, because the
fit-on-train rule covers tokenizers too (`train_from_iterator` *is* a
`.fit()`) — and a typo decomposes into familiar pieces instead of vanishing.
Hand that tokenizer to `TfidfVectorizer` as the `analyzer`, rebuild the
tensors, and reuse `fit()` unchanged.

In [31]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

# Week 1's byte-level BPE, trained on the TRAINING tickets only.
bpe = Tokenizer(models.BPE())
bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
bpe.train_from_iterator(train_df["text"], trainers.BpeTrainer(
    vocab_size=5000, initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))

# The mechanism, on real typos from this corpus:
for w in [" tracking", " trackign", " package", " pacakge"]:
    print(f"{w.strip():10s} -> {bpe.encode(w).tokens}")

# Same TF-IDF recipe, new tokenizer: analyzer= replaces the default regex.
vec_bpe = TfidfVectorizer(analyzer=lambda text: bpe.encode(text).tokens, min_df=3)
Xb_train_t = to_tensor(vec_bpe.fit_transform(train_df["text"]))   # fit: TRAIN only
Xb_val_t   = to_tensor(vec_bpe.transform(val_df["text"]))
n_bpe = Xb_train_t.shape[1]
print(f"\n{N_FEATURES} word columns -> {n_bpe} subword columns")

# New data, same loop: batches and validation tensors are just arguments now.
train_dl_bpe = DataLoader(TensorDataset(Xb_train_t, y_train_t),
                          batch_size=200, shuffle=True)
subword = make_model(n_features=n_bpe)
bpe_val, _ = fit(subword, data=train_dl_bpe, X_val=Xb_val_t)

results.append({"variant": "BPE-5k tokenizer", "val_accuracy": bpe_val,
                "val_macro_F1": val_macro_f1(subword, X=Xb_val_t)})
print(f"\nBPE-5k: val accuracy {bpe_val:.4f}   "
      f"val macro F1 {val_macro_f1(subword, X=Xb_val_t):.4f}\n")

print(pd.DataFrame(results).round(4).to_string(index=False))




tracking   -> ['Ġtracking']
trackign   -> ['Ġtrack', 'ign']
package    -> ['Ġpackage']
pacakge    -> ['Ġp', 'ac', 'ak', 'ge']



636 word columns -> 750 subword columns

BPE-5k: val accuracy 0.9266   val macro F1 0.9210

            variant  val_accuracy  val_macro_F1
      baseline (v4)        0.9198        0.9126
class-weighted loss        0.9164        0.9082
        dropout 0.3        0.9181        0.9111
       hidden = 512        0.9198        0.9128
   BPE-5k tokenizer        0.9266        0.9210


Read the table the way Week 3 taught you. One change *lost*: class
weighting dipped both metrics, because our smallest queue is ~12% of
tickets, not 1% — there was no sacrificed class to rescue. That is not a
wasted experiment; you turned a knob sklearn does not have, measured what it
did, and can say so. And one change clearly *won* — the prep-side one: the
BPE tokenizer posted the best accuracy and macro F1 in the table, with
Week 1's subwords fixing what Week 3's word regex was throwing away. The
model-side changes moved the needle less than the data-side one did —
**prep beats capacity** again. Keep the honest caveat in reach: on 586
validation tickets these differences are about one standard error, so treat
the ordering as evidence, not proof. And **every number here is a
validation number** — we used validation to compare variants, which is
exactly its job. Only a final, chosen model earns a run against the test
set, and we already spent ours on the baseline. (Your assignment's Task 5
asks for exactly this kind of measured, one-at-a-time change — a
self-trained tokenizer qualifies as "one improvement of your own design.") On a corpus
this size, prep and honest evaluation move the needle more than capacity —
the lecture's "prep beats capacity" lesson, reproduced in your own loop.

---
## Where you now stand

| you knew (sklearn) | you now own (PyTorch) |
|---|---|
| `MLPClassifier(hidden_layer_sizes=(128,))` | `nn.Sequential(nn.Linear(636, 128), nn.ReLU(), nn.Linear(128, 5))` |
| `.fit(X, y)` | the five-line loop + DataLoader batches |
| `early_stopping=True` | your patience counter + `state_dict` rollback |
| `.predict(X)` | `model(X).argmax(dim=1)` (under `no_grad`, in `eval` mode) |
| `verbose=True` | your own `print` — log anything you like |
| *(no equivalent)* | class-weighted loss, a tokenizer you trained yourself, dropout where you choose, layers you invent |

Week 4 keeps the loss, the optimizer, and this exact loop — and only swaps
what is inside the model: layers that read text as a *sequence* instead of a
bag of numbers. The loop you built today is the one you will use for the rest
of the course.

*Companions: `ism6564-week03-pytorch-intro.ipynb` (sklearn ↔ PyTorch, side by
side), `pytorch-primer.ipynb` (tensors and autograd drills), and lecture
notebook §4.4 (this same road on SVD-compressed features).*